# QMSSGR-5069: Take-home-exercise-4 Model Deployment

Haoxuan Lu

CUID: hl3921

Date: Apr.23, 2026

#### Homework #5: model deployment Instructions: Using the F1 dataset, build a predictive model and log it in MLflow and write the ML model predictions into a database.

#### 1. [20 pts] Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.

In [0]:
# --------------------------------------------------
# Create two new tables to store model predictions
# --------------------------------------------------

# Use default database
spark.sql("USE default")

# Drop old tables if they exist
spark.sql("DROP TABLE IF EXISTS model1_predictions")
spark.sql("DROP TABLE IF EXISTS model2_predictions")

# ----------------------------------------
# Table for Model 1 Predictions
# ----------------------------------------
spark.sql("""
CREATE TABLE model1_predictions (
    raceId INT,
    driverId INT,
    constructorId INT,
    actual_position INT,
    predicted_position DOUBLE,
    model_name STRING,
    created_at TIMESTAMP
)
USING DELTA
""")

# ----------------------------------------
# Table for Model 2 Predictions
# ----------------------------------------
spark.sql("""
CREATE TABLE model2_predictions (
    raceId INT,
    driverId INT,
    constructorId INT,
    actual_position INT,
    predicted_position DOUBLE,
    model_name STRING,
    created_at TIMESTAMP
)
USING DELTA
""")

# Verify tables created
spark.sql("SHOW TABLES").show(truncate=False)

#### 2. [30 pts] Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import tempfile
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [0]:
# Load F1 results data
df = spark.read.csv(
    "dbfs:/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
# Prepare data
df_pd = df.toPandas()
df_pd = df_pd.replace("\\N", np.nan)

selected_cols = [
    "raceId",
    "driverId",
    "constructorId",
    "grid",
    "laps",
    "fastestLap",
    "rank",
    "fastestLapSpeed",
    "points",
    "positionOrder"
]

df_model = df_pd[selected_cols].copy()

for col in df_model.columns:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

df_model = df_model.dropna()

X = df_model.drop("positionOrder", axis=1)
y = df_model["positionOrder"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
display(df_model.head())

In [0]:
# set mlflow experiments
mlflow.set_experiment("/Shared/HW5_F1_Model_Deployment")

In [0]:
# helper function for mlflow logging
def log_model_experiment(model, model_name, params, X_train, X_test, y_train, y_test):
    with mlflow.start_run(run_name=model_name):
        
        # Train model
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        # Metrics
        mae = mean_absolute_error(y_test, preds)
        mse = mean_squared_error(y_test, preds)
        rmse = mse ** 0.5
        r2 = r2_score(y_test, preds)
        
        # Log params
        mlflow.log_params(params)
        
        # Log metrics
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        
        # Log model
        mlflow.sklearn.log_model(model, artifact_path=model_name)
        
        with tempfile.TemporaryDirectory() as tmpdir:
            
            # Artifact 1: Actual vs Predicted plot
            plt.figure(figsize=(6, 4))
            plt.scatter(y_test, preds, alpha=0.5)
            plt.xlabel("Actual Position")
            plt.ylabel("Predicted Position")
            plt.title(f"{model_name}: Actual vs Predicted")
            plot1_path = os.path.join(tmpdir, f"{model_name}_actual_vs_predicted.png")
            plt.savefig(plot1_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(plot1_path)
            
            # Artifact 2: Feature Importance plot
            importances = pd.Series(model.feature_importances_, index=X_train.columns)
            importances = importances.sort_values(ascending=False)
            
            plt.figure(figsize=(8, 4))
            importances.plot(kind="bar")
            plt.title(f"{model_name}: Feature Importance")
            plt.ylabel("Importance")
            plot2_path = os.path.join(tmpdir, f"{model_name}_feature_importance.png")
            plt.savefig(plot2_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(plot2_path)
            
            # Extra artifact: predictions CSV
            pred_df = X_test.copy()
            pred_df["actual_position"] = y_test.values
            pred_df["predicted_position"] = preds
            
            csv_path = os.path.join(tmpdir, f"{model_name}_predictions.csv")
            pred_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)
        
        print(f"{model_name} finished.")
        print(f"MAE:  {mae:.4f}")
        print(f"MSE:  {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"R2:   {r2:.4f}")

In [0]:
# model 1: random forest
rf_params = {
    "model_type": "RandomForestRegressor",
    "n_estimators": 150,
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": 42
}

rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

log_model_experiment(
    model=rf_model,
    model_name="random_forest_model",
    params=rf_params,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

In [0]:
# model 2: gradient boosting
gb_params = {
    "model_type": "GradientBoostingRegressor",
    "n_estimators": 150,
    "learning_rate": 0.05,
    "max_depth": 3,
    "random_state": 42
}

gb_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

log_model_experiment(
    model=gb_model,
    model_name="gradient_boosting_model",
    params=gb_params,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

In [0]:
# view experiment runs
experiment = mlflow.get_experiment_by_name("/Shared/HW5_F1_Model_Deployment")
experiment_id = experiment.experiment_id

runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.r2 DESC"]
)

display(runs_df)

In [0]:
# show only important columns
summary_df = runs_df[[
    "run_id",
    "tags.mlflow.runName",
    "metrics.mae",
    "metrics.mse",
    "metrics.rmse",
    "metrics.r2",
    "params.model_type"
]]

display(summary_df)

#### 3. [30 pts] For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.

In [0]:
from pyspark.sql import functions as F

# -----------------------------
# Model 1: Random Forest
# -----------------------------
rf_preds = rf_model.predict(X_test)

model1_pred_pd = X_test.copy()
model1_pred_pd["actual_position"] = y_test.values
model1_pred_pd["predicted_position"] = rf_preds
model1_pred_pd["model_name"] = "random_forest_model"

# Convert to Spark DataFrame
model1_pred_spark = spark.createDataFrame(model1_pred_pd)

# Add timestamp
model1_pred_spark = model1_pred_spark.withColumn("created_at", F.current_timestamp())

# Keep only columns that match the table schema
model1_pred_spark = model1_pred_spark.select(
    "raceId",
    "driverId",
    "constructorId",
    "actual_position",
    "predicted_position",
    "model_name",
    "created_at"
)

# Write into prediction table
model1_pred_spark.write.mode("append").saveAsTable("default.model1_predictions")

print("Model 1 predictions written to default.model1_predictions")
display(model1_pred_spark)

In [0]:
# -----------------------------
# Model 2: Gradient Boosting
# -----------------------------
gb_preds = gb_model.predict(X_test)

model2_pred_pd = X_test.copy()
model2_pred_pd["actual_position"] = y_test.values
model2_pred_pd["predicted_position"] = gb_preds
model2_pred_pd["model_name"] = "gradient_boosting_model"

# Convert to Spark DataFrame
model2_pred_spark = spark.createDataFrame(model2_pred_pd)

# Add timestamp
model2_pred_spark = model2_pred_spark.withColumn("created_at", F.current_timestamp())

# Keep only columns that match the table schema
model2_pred_spark = model2_pred_spark.select(
    "raceId",
    "driverId",
    "constructorId",
    "actual_position",
    "predicted_position",
    "model_name",
    "created_at"
)

# Write into prediction table
model2_pred_spark.write.mode("append").saveAsTable("default.model2_predictions")

print("Model 2 predictions written to default.model2_predictions")
display(model2_pred_spark)

In [0]:
print("Model 1 table:")
display(spark.sql("SELECT * FROM default.model1_predictions LIMIT 10"))

print("Model 2 table:")
display(spark.sql("SELECT * FROM default.model2_predictions LIMIT 10"))

In [0]:
spark.sql("""
SELECT 'model1_predictions' AS table_name, COUNT(*) AS row_count
FROM default.model1_predictions
UNION ALL
SELECT 'model2_predictions' AS table_name, COUNT(*) AS row_count
FROM default.model2_predictions
""").show()